In [46]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("MULTI_SERVICES_ROUTER_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "Multi-Services Router"
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["GITHUB_TOKEN"] = os.getenv("GITHUB_TOKEN")
os.environ["GITHUB_PERSONAL_ACCESS_TOKEN"] = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")
os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")
os.environ["LLAMA_3_3_70B_INSTRUCT_API_KEY"] = os.getenv("LLAMA_3_3_70B_INSTRUCT_API_KEY")


from langchain_openai import ChatOpenAI
from typing import Annotated, TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from contextlib import AsyncExitStack
from langchain_core.tracers.context import tracing_v2_enabled
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from glob import glob
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langchain_core.tools.retriever import create_retriever_tool
from langgraph.graph import MessagesState
from langgraph_supervisor import create_supervisor


# Agents

## Searcher Reflection Agent

#### RAG

In [2]:
# Expand glob pattern to actual file paths and load each PDF separately
pdf_paths = glob("./RAG PDFs/*.pdf")
if not pdf_paths:
    raise FileNotFoundError("No PDF files found in ./RAG PDFs/*.pdf")

all_docs = []
for path in pdf_paths:
    loader = PyMuPDFLoader(path)
    all_docs.extend(loader.load())

# Target documents 
target_docs = [doc for doc in all_docs]

# Splitter - Increase chunk_size
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
doc_splits = text_splitter.split_documents(target_docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

vectorstore = Chroma.from_documents(
    documents=doc_splits,
    embedding=embeddings,
    collection_name="deep-learning-rag"
)

# Fetch more context chunks
retriever_pdf = vectorstore.as_retriever(search_kwargs={"k": 5})


retriever_tool = create_retriever_tool(
    retriever_pdf,
    "deep-learning-rag-retriever",
    "Search and return information about basic deep learning topics.",
)

retriever_tool.invoke("What is deep learning?")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4594.89it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'210\n\x0c\x0c\x0c Deep learning\n\n8\nCHAPTER 1\nWhat is deep learning?\nAs you can see in figure 1.6, the network transforms the digit image into representa-\ntions that are increasingly different from the original image and increasingly informa-\ntive about the final result. You can think of a deep network as a multistage information-\ndistillation process, where information goes through successive filters and comes out\nincreasingly purified (that is, useful with regard to some task).\nSo that’s what deep learning is, technically: a multistage way to learn data representa-\ntions. It’s a simple idea—but, as it turns out, very simple mechanisms, sufficiently\nscaled, can end up looking like magic. \n1.1.5\nUnderstanding how deep learning works, in three figures\nAt this point, you know that machine learning is about mapping inputs (such as\nimages) to targets (such as the label “cat”), which is done by observing many examples\nof input and targets. You also know that deep neural net

#### Search Engine

In [3]:
tavily_tool = TavilySearchResults(max_results=3)

/tmp/ipykernel_11453/2877639967.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=3)


### Create Author Agent

In [ ]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    binary_score: str
    critics_feedback: str

In [87]:
from langchain.chat_models import init_chat_model
from langchain_nvidia_ai_endpoints import ChatNVIDIA

response_model = ChatGroq(model="qwen/qwen3-32b", temperature=0)


async def author_node(state: State) -> dict:
    """
    Answer the user question using Tavily Search for general knowledge questions and the retriever tool for technical questions related to deep learning. If the question is a greeting or a general knowledge question that can be answered without assistance, respond directly. If you don't have the information to answer the question, say something like 'I don't have that information'.
    """

    AUTHOR_PROMPT = SystemMessage(content="""
        Eres un experto en Deep Learning y Orquestación de Servicios.
        Tu tarea es responder consultas técnicas con precisión quirúrgica.

        REGLAS DE ORO:
        1. Si es la primera vez que respondes: Usa 'retriever_tool' para conceptos técnicos y 'tavily_tool' para info general.
        2. Si recibes una CRÍTICA del revisor: NO uses herramientas de nuevo a menos que falte información. Ajusta tu respuesta previa siguiendo las instrucciones de la crítica.
        3. Formato: No menciones que eres una IA ni que estás siendo evaluado. Entrega la respuesta final directamente.
        4. Idioma: Responde siempre en el idioma del usuario.
    """)

    print(state)
    prompt = [AUTHOR_PROMPT] + state["messages"] + state.get("critics_feedback", [])
    llm_with_tools = response_model.bind_tools(tools=[retriever_tool, tavily_tool])

    response = await (llm_with_tools.ainvoke(prompt))
    return {"messages": [response]}

#### Test

In [74]:
input = {"messages": [{"role": "user", "content": "how's the Australian Open last winner?"}]}
response = await author_node(input)
response["messages"][-1].pretty_print()

{'messages': [{'role': 'user', 'content': "how's the Australian Open last winner?"}]}
================================== Ai Message ==================================

The most recent Australian Open winners are Novak Djokovic (men's singles) and Aryna Sabalenka (women's singles), who won the 2023 edition held in January 2023. The 2024 tournament has not yet taken place as of October 2023.


#### Refelction Node

In [83]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

class GradeDocuments(BaseModel):
    """Grade answer completion using a binary score."""
    score: Literal["yes", "no"] = Field(
        description="Asign binary score to the answer where 'yes' means complete and 'no' means incomplete."
    )
    critique: str = Field(
        description="Detailed critique and recommendations for the user's submission."
    )

reflection_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Eres un Revisor Académico de élite. Tu objetivo es asegurar que la respuesta del Autor sea perfecta.
                Evalúa la respuesta del Autor basada en el historial de mensajes.

                CRITERIOS DE EVALUACIÓN:
                - Precisión técnica (¿Es correcto lo que dice sobre Deep Learning?).
                - Completitud (¿Responde a todo lo que el usuario pidió?).
                - Claridad y Estructura.

                Si la respuesta es excelente, marca score: "yes".
                Si falta algo o se puede mejorar, marca score: "no" y detalla los cambios necesarios en 'critique'.""",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

async def reflection_node(state: State):
    # Format the messages to not annidate the system prompt with the user messages
    messages_to_grade = reflection_prompt.format_messages(messages=state["messages"])
    response = await response_model.with_structured_output(GradeDocuments).ainvoke(messages_to_grade)
    state["critics_feedback"] = response.critique
    
    # Return both the binary score and the critique in the messages
    # The binary score can be used for routing decisions, while the critique provides valuable feedback to the user.
    return {
        "binary_score": response.score,
        "critics_feedback": response.critique,
    }

### Searcher Subgraph

In [89]:
cant = 0
def should_continue(state: State) -> Literal["generator", "end"]:
    """Determine whether to generator the workflow or end it."""
    if (len(state["messages"]) < 6):  # Allow up to 3 iterations
        return "repeat"
    return "end"

In [90]:
def build_searcher_graph(): # -> CompiledGraph      

    workflow = StateGraph(State)
    workflow.add_node("generator", author_node)
    workflow.add_node("tools_node", ToolNode(tools=[retriever_tool, tavily_tool]))
    workflow.add_node("revisor", reflection_node)

    workflow.add_edge(START, "generator")    

    workflow.add_conditional_edges("generator", tools_condition,
        {
            "tools": "tools_node",
            END: "revisor"
        }
    )

    workflow.add_edge("tools_node", "generator")  
    workflow.add_conditional_edges("revisor", should_continue,
        {
            "end": END,
            "repeat": "generator"
        }
    )

    return workflow.compile(name="SearcherGraph")

#### Test

In [91]:

graph = build_searcher_graph()


inputs = {"messages": [HumanMessage(content="What is a loss function?")]}

memory = MemorySaver()
config= {
    "configurable": {
        "thread_id": "thread1"
    }
}
response = await graph.ainvoke(inputs, config=config)
response["messages"][-1].pretty_print()

{'messages': [HumanMessage(content='What is a loss function?', additional_kwargs={}, response_metadata={}, id='209c18a1-0975-4a21-9090-28a9ad6f469d')]}
{'messages': [HumanMessage(content='What is a loss function?', additional_kwargs={}, response_metadata={}, id='209c18a1-0975-4a21-9090-28a9ad6f469d'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking, "What is a loss function?" Let me start by recalling my knowledge. A loss function is a critical component in machine learning models, especially in training neural networks. It quantifies how well the model\'s predictions match the actual data. The goal during training is to minimize this loss.\n\nFirst, I need to define it clearly. The loss function measures the difference between predicted and actual values. Common examples include Mean Squared Error for regression and Cross-Entropy for classification. But I should explain why it\'s important—like guiding the optimization process through gradient d

## Bots Agent

In [19]:
# Mantenemos el cliente fuera
bots_client = MultiServerMCPClient(
    {
        "BotsAgent": {
            "url": "http://localhost:8001/sse",
            "transport": "sse"
        }
    }
)

def build_bots_workflow(bot_tools):
    bot_model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).bind_tools(tools=bot_tools)

    async def bot_node(state: MessagesState):
        sys_msg = SystemMessage(content=(
            "You are an assistant that can call tools related to Bots to assist users. "
            "Given a user question, decide which tool to call and with what arguments. "
            "If the question is not related to any tool, respond directly to the user. "
            "Dont report any IP if the user asks about it."
            "If you have the information to answer the user's question without calling a tool, respond directly to the user. "
        ))
        prompt = [sys_msg] + state["messages"]
        response = await bot_model.ainvoke(prompt)
        return {"messages": [response]}

    workflow = StateGraph(MessagesState)
    workflow.add_node("bot_agent", bot_node)
    workflow.add_node("tools", ToolNode(bot_tools))
    
    workflow.add_edge(START, "bot_agent")
    workflow.add_conditional_edges("bot_agent", tools_condition)
    workflow.add_edge("tools", "bot_agent")
    
    return workflow.compile(name="BotsGraph")


#### Test

In [11]:
# La función de ejecución maneja el ciclo de vida de los recursos
async def run_bots_orchestrator(inputs):
    # Open the connection to MCP and load tools
    async with AsyncExitStack() as stack:
        session = await stack.enter_async_context(bots_client.session("BotsAgent"))
        bot_tools = await load_mcp_tools(session)
        
        # Build the workflow graph with the loaded tools
        bots_graph = build_bots_workflow(bot_tools)
        
        # Execute the graph while the connection is still alive
        result = await bots_graph.ainvoke(inputs)
        
        for msg in result["messages"]:
            msg.pretty_print()

await run_bots_orchestrator({"messages": [{"role": "user", "content": "Exists the IP 192.168.1.1?"}]})

================================ Human Message =================================

Exists the IP 192.168.1.1?
================================== Ai Message ==================================
Tool Calls:
  get_bot_by_ip (ycjdepwsr)
 Call ID: ycjdepwsr
  Args:
    ip: 192.168.1.1
================================= Tool Message =================================
Name: get_bot_by_ip

[{'type': 'text', 'text': '{"ip":"192.168.1.1","last_attack":"2026-04-29T18:17:53.812018","blocked_window":7,"num_attacks":34}', 'id': 'lc_6ade478d-d2ab-4e8d-b275-34e968d61088'}]
================================== Ai Message ==================================

Yes, the IP 192.168.1.1 exists and has been recorded in the database. It was last seen attacking on April 29, 2026, and has been involved in a total of 34 attacks.


## Github Agent

### State

In [8]:
from langchain_core.messages import BaseMessage, ToolMessage

class GithubState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    commit_message: str

In [10]:
import os

# This avoids system's "ammnestia"
system_env = dict(os.environ)

# Merge system environment with GitHub token for the GitHub server
github_env = {**system_env, "GITHUB_PERSONAL_ACCESS_TOKEN": os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")}

path = "/home/santi/Documentos/LangGraph/"

# MCP Client Configuration
client = MultiServerMCPClient(
    {
        "filesystem": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", path],
            "transport": "stdio",
            "env": system_env # Optional
        },
        "github": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-github"],
            "env": github_env, # Merged environment with GitHub token
            "transport": "stdio",
        }
    }
)

In [25]:
import warnings
warnings.filterwarnings("ignore", message=".*is not supported in schema.*")
from langgraph.checkpoint.memory import MemorySaver

async def build_github_workflow(tools):
    github_model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0).bind_tools(tools=tools)

    async def github_node(state: GithubState):
        WORKSPACE_PATH = path # Tu variable de entorno
        DEFAULT_REPO = "santiFie/LangGraph"

        sys_msg = f"""
        # ROLE
        You are a Senior Platform Engineer assistant specialized in GitHub workflows and Local Filesystem operations.

        # CONTEXT
        - **Root Path:** {WORKSPACE_PATH}
        - **Primary Repository:** {DEFAULT_REPO} (Always assume this repo unless specified).

        # OPERATIONAL GUIDELINES
        1. **Tool Selection:** 
        - Use `filesystem` tools for local file operations.
        - Use `github` tools ONLY for remote repository interactions.
        2. **Path Resolution:** If only a filename is provided, use directory listing tools to find the relative path from the workspace root before taking action.
        3. **Search Protocol:** Use fully qualified identifiers (e.g., `repo:{DEFAULT_REPO}`) for searches. Only search if explicitly requested.

        # CRITICAL CONSTRAINTS (Read Carefully)
        - **Commit Messages:** NEVER generate, guess, or placeholder a commit message. If missing, leave the argument null/empty to trigger human intervention.
        - **Tool Fallback:** If a file/repo is missing or the request is ambiguous, ask for clarification instead of guessing tool arguments.
        - **Direct Response:** If the answer is known or the request is non-technical, respond directly without calling tools.

        # TASK
        Analyze the user's request and determine the next logical step.
        """
        prompt = [sys_msg] + state["messages"]

        if state.get("commit_message"):
            print("Github Node: Commit message detected in state.")
            prompt.append(SystemMessage(content=f"User has provided this commit message: {state['commit_message']}"))
        
        response = await github_model.ainvoke(prompt)
        return {"messages": [response]}
    
    def human_committer(state: GithubState):
        
        commit_message = state.get("commit_message", [""])
        if not commit_message:
        
            return {"messages": [HumanMessage(content=f"Please provide a commit message for the following commit: {commit_message[-1]}")]}
        
        print("Human Committer: Commit message received, proceeding with the workflow: ", commit_message)
        return {"messages": [HumanMessage(content=f"Commit message received: {commit_message}")]}
        

    def should_continue(state: GithubState):
        last_message = state["messages"][-1]

        if last_message.tool_calls:
            for call in last_message.tool_calls:
                if call["name"] == "create_or_update_file":
                    if not call["args"].get("message"):
                        return "human_committer"
            return "tools"
        return END
    
    workflow = StateGraph(GithubState)

    workflow.add_node("github_agent", github_node)
    workflow.add_node("tools", ToolNode(tools))
    workflow.add_node("human_committer", human_committer)

    workflow.add_edge(START, "github_agent")
    workflow.add_conditional_edges("github_agent", should_continue, {
        "human_committer": "human_committer",
        "tools": "tools",
        END: END
    })
    workflow.add_edge("tools", "github_agent")
    workflow.add_edge("human_committer", "github_agent")

    memory = MemorySaver()
    
    return workflow.compile(
        checkpointer=memory,
        interrupt_before=["human_committer"],
        name="GitHubGraph"
    )

#### Test

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="*")

async def create_():
    async with AsyncExitStack() as stack:
        # Connect to both servers
        github_session = await stack.enter_async_context(client.session("github"))
        filesystem_session = await stack.enter_async_context(client.session("filesystem"))

        # Load tools from both sessions
        github_tools = await load_mcp_tools(github_session)
        filesystem_tools = await load_mcp_tools(filesystem_session)

        # Combine tools into a single list
        all_tools = github_tools + filesystem_tools

        # Define the workflow with the combined tools
        workflow = await build_github_workflow(all_tools)

        # Example input_message to the workflow
        input_message = {"messages": [HumanMessage(content="Create a file named 'test.txt' with the content 'Hello World' in the filesystem, then commit it in the repository 'santiFie/LangGraph'")]}

        # Execute the workflow
        config = {"configurable": {"thread_id": "1"}}
        # graph = await workflow.ainvoke(input_message, config)

        async for event in workflow.astream(input_message, config):
            for key in event:
                print(f"Node Executed: {key}")
        
        # Check if PAUSED waiting for human input_message
        state = workflow.get_state(config)
        if state.next and "human_committer" in state.next:
            print("Workflow is paused, waiting for commit message.")

            commit_message = input("Please enter the commit message: ")

            # Update the workflow state with the provided commit message
            workflow.update_state(config, {"commit_message": commit_message})

            async for event in workflow.astream(None, config):
                for key in event:
                    print(f"Node Executed: {key}")

        final_state = workflow.get_state(config)
        print("Final State:", final_state)

await create_()

Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is

Node Executed: github_agent
Node Executed: tools
Node Executed: github_agent
Node Executed: __interrupt__
Workflow is paused, waiting for commit message.


Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is

Human Committer: Commit message received, proceeding with the workflow:  This is a Human-in-the-loop commit message
Node Executed: human_committer
Github Node: Commit message detected in state.
Node Executed: github_agent


Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is

Node Executed: tools
Github Node: Commit message detected in state.
Node Executed: github_agent
Final State: StateSnapshot(values={'messages': [HumanMessage(content="Create a file named 'test.txt' with the content 'Hello World' in the filesystem, then commit it in the repository 'santiFie/LangGraph'", additional_kwargs={}, response_metadata={}, id='1d6d62b0-5d89-4378-b203-d241e29f676e'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'write_file', 'arguments': '{"content": "Hello World", "path": "test.txt"}'}, '__gemini_function_call_thought_signatures__': {'4a6ba4c6-e0b4-4d87-a633-a65023b41954': 'EjQKMgEMOdbHsQbhtOgoUmkEwM8+ZNTI9a6uA5Xf3Q+tUuTPGFKakf3461/8o+PTUtYx5nQ6'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dee42-6027-75f3-a306-3f9835e5fb81-0', tool_calls=[{'name': 'write_file', 'args': {'content': 'Hello World', 'path': 'test.txt'}, 'id': 

# Supervisor

## Create Supervisor

In [15]:
supervisor_model = ChatOpenAI(
                model="z-ai/glm-5.1",
                api_key=os.getenv("NVIDIA_API_KEY"),
                base_url="https://integrate.api.nvidia.com/v1", # NVIDIA's API URL
                temperature=0.0,
            )

## Graph

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# 1) Compile your subgraphs (make sure these builders exist in earlier cells)
# - searcher_graph: RAG + search tools
# - bots_graph: MCP Bots agent
# - github_graph: MCP GitHub + filesystem agent

async def main(user_input: str):

    async with AsyncExitStack() as stack:

        # 1) Open all the sessions
        bots_session = await stack.enter_async_context(bots_client.session("BotsAgent"))
        github_session = await stack.enter_async_context(client.session("github"))
        filesystem_session = await stack.enter_async_context(client.session("filesystem"))

        # 2) Load tools for each subgraph
        bot_tools = await load_mcp_tools(bots_session)
        github_tools = await load_mcp_tools(github_session)
        filesystem_tools = await load_mcp_tools(filesystem_session)

        # 3) Build/compile subgraphs with their respective tools
        searcher_graph = build_searcher_graph()  # Already compiled in the function
        bots_graph = build_bots_workflow(bot_tools)  # Already compiled in the function
        github_graph = await build_github_workflow(github_tools + filesystem_tools)  # Already compiled in the function

        # 4) Build the supervisor graph with the compiled subgraphs
        memory = MemorySaver()
        supervisor_graph = create_supervisor(
            model=supervisor_model,
            agents=[searcher_graph, bots_graph, github_graph],
            prompt=(
                "You are a supervisor that routes tasks to specialized subgraphs: "
                "- 'searcher' for answering questions using retrieved documents about deep learning or searching in the internet, "
                "- 'bots' for answering questions related to bots attacks logs, "
                "- 'github' for answering questions related to GitHub repositories and filesystem operations. "
            )
        ).compile(checkpointer=memory)

        # Execute
        config = {"configurable": {"thread_id": "1"}}
        input_message = {"messages": [HumanMessage(content=user_input)]}

        async for event in supervisor_graph.astream(input_message, config):
            for key in event:
                print(f"Node Executed: {key}")

        state = supervisor_graph.get_state(config)
        if state.next:
            print("Workflow is paused, waiting for commit message.")
            commit_message = input("Please enter the commit message: ")

            #Find the specific subgraph state that is waiting for the commit message (in this case, github_graph)
            subgraph_config = config
            if hasattr(state, "tasks") and state.tasks:
                for task in state.tasks:
                    if hasattr(task, "state") and task.state and task.state.next:
                        subgraph_config = task.state.config
                        break
            
            # Update the specific subgraph state with the commit message
            supervisor_graph.update_state(subgraph_config, {"commit_message": commit_message})

            # Retake the execution of the subgraph that was paused
            async for event in supervisor_graph.astream(None, config):
                for key in event:
                    print(f"Node Executed: {key}")
                    
        print("Fin del proceso global.")

await main("Create a file named 'test2.txt' and commit without specifying a message")


Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is

================================ Human Message =================================

Create a file named 'bots_report.txt' and write in it the 5 bots more attacked the app. Then, make a commit with the message 'Added bots report by a github agent' and push the changes to GitHub
================================== Ai Message ==================================
Name: supervisor

I'll help you with that! I need to first get the top 5 most attacked bots from the logs, and then create the file, commit, and push to GitHub. Let me start by fetching the bots attack data.
Tool Calls:
  transfer_to_botsgraph (chatcmpl-tool-b42dd5838a105499)
 Call ID: chatcmpl-tool-b42dd5838a105499
  Args:
================================= Tool Message =================================
Name: transfer_to_botsgraph

Successfully transferred to BotsGraph
================================== Ai Message ==================================

Here are the top 5 most attacked bots:
1. 192.168.1.2 - 93 attacks
2. 192.168.1.11 - 93